# WeatherNext 2 — ERA5 Historical TC Reforecast

Adapted from Google DeepMind's `wn2_demo.ipynb` for **reforecasting a
historical tropical cyclone from ERA5 reanalysis**, instead of the bundled
HRES-forecast sample the original demo ships with.

**What's different from the original demo:**
- Input weather state is pulled from **WeatherBench2's ERA5 zarr** for a
  storm/date you choose from IBTrACS (within ERA5's 1959-2023 coverage),
not the fixed 2024-10-07 HRES sample.
- The **verification target is also real ERA5** for the forecast window (not
  the bundled forecast file's own later steps), so the comparison plots and
  CRPS score are a genuine reforecast-vs-reanalysis check.
- This notebook still runs **one storm at a time**, same as the original demo.
  For a resumable loop over many historical storms, see the companion
  `reforecast_pipeline.py` script.

**Before running at scale, two things are worth validating empirically**
(see the discussion in chat this notebook came out of):
1. Whether ERA5 initialization gives comparable track/intensity skill to the
   bundled HRES sample for a known case (`WeatherNextCyclones_Mini` isn't
   documented either way).
2. That the checkpoint's training cutoff (2023 or 2022 depending on which
   `_Mini_<year>` weights you use) doesn't overlap the storm you're
   reforecasting, if you care about out-of-sample skill.

## Installation and Initialization

In [ ]:
# @title Upgrade packages (kernel needs to be restarted after running this cell).
%pip install -U importlib_metadata


In [ ]:
# @title Pip install repo and dependencies, plus zarr/gcsfs for reading ERA5 from GCS.
%pip install --upgrade https://github.com/google-deepmind/weathernext/archive/master.zip --ignore-requires-python
%pip install -U zarr gcsfs
# This is required due to outdated jax and libtpu versions in Colab TPU images.
%pip uninstall -y libtpu libtpu-nightly
%pip install -U "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html


     | 2.5 MB 10.6 MB/s 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/google-deepmind/xarray_jax.git (to revision v0.1.1) to /tmp/pip-install-golvtm85/gdm-xarray-jax_b05beebfd64f4084abdedfde02d095b6
  Running command git clone --filter=blob:none --quiet https://github.com/google-deepmind/xarray_jax.git /tmp/pip-install-golvtm85/gdm-xarray-jax_b05beebfd64f4084abdedfde02d095b6
  Running command git checkout -q 0291d0c8e91ca04b00f5f01ea29a53c0e575eeee
  Resolved https://github.com/google-deepmind/xarray_jax.git to commit 0291d0c8e91ca04b00f5f01ea29a53c0e575eeee
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━

Found existing installation: libtpu 0.0.21.1
Uninstalling libtpu-0.0.21.1:
  Successfully uninstalled libtpu-0.0.21.1
Looking in links: https://storage.googleapis.com/jax-releases/libtpu_releases.html
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.4/221.4 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 102.9 MB/s eta 0:00:00
  Attempting uninstall: jaxlib
    Found existing installation: jaxlib 0.7.2
    Uninstalling jaxlib-0.7.2:
      Successfully uninstalled jaxlib-0.7.2
  Attempting uninstall: jax
    Found existing installation: jax 0.7.2
    Uninstalling jax-0.7.2:
      Successfully uninstalled jax-0.7.2


In [ ]:
# @title Imports

import dataclasses
import datetime
import math
from google.colab import auth
from google.cloud import storage
from typing import Optional
import haiku as hk
from IPython.display import HTML
from IPython import display
import ipywidgets as widgets
import jax
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
import xarray
import pandas as pd

import xarray_jax
from weathernext.utils import rollout
from weathernext.utils import checkpoint
from weathernext.utils import data_utils
from weathernext.utils import xarray_tree
from weathernext.utils import fiddle_config_io
from weathernext.weathernext2 import fgn
from weathernext.cyclones import constants as cyclone_constants
from weathernext.cyclones import direct_tracker_6h_v1_config
from weathernext.cyclones import ibtracs_netcdf_to_csv


/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:90: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [ ]:
# @title Plotting functions (unchanged from the original demo)

def select(
    data: xarray.Dataset,
    variable: str,
    level: Optional[int] = None,
    max_steps: Optional[int] = None
    ) -> xarray.Dataset:
  data = data[variable]
  if "batch" in data.dims:
    data = data.isel(batch=0)
  if max_steps is not None and "time" in data.sizes and max_steps < data.sizes["time"]:
    data = data.isel(time=range(0, max_steps))
  if level is not None and "level" in data.coords:
    data = data.sel(level=level)
  return data

def scale(
    data: xarray.Dataset,
    center: Optional[float] = None,
    robust: bool = False,
    ) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:
  vmin = np.nanpercentile(data, (2 if robust else 0))
  vmax = np.nanpercentile(data, (98 if robust else 100))
  if center is not None:
    diff = max(vmax - center, center - vmin)
    vmin = center - diff
    vmax = center + diff
  return (data, matplotlib.colors.Normalize(vmin, vmax),
          ("RdBu_r" if center is not None else "viridis"))

def plot_data(
    data: dict[str, xarray.Dataset],
    fig_title: str,
    plot_size: float = 5,
    robust: bool = False,
    cols: int = 4,
    cyclone_dfs: Optional[dict[str, pd.DataFrame]] = None,
    init_time: Optional[np.datetime64] = None,
    ) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:

  first_data = next(iter(data.values()))[0]
  max_steps = first_data.sizes.get("time", 1)
  assert all(max_steps == d.sizes.get("time", 1) for d, _, _ in data.values())

  if cyclone_dfs:
    lead_times = first_data.coords["time"].values[:max_steps]
    valid_times = pd.Timestamp(init_time) + pd.to_timedelta(lead_times)

  lat_coords = first_data.coords["lat"].values
  lon_coords = first_data.coords["lon"].values
  lat_min, lat_max = lat_coords.min(), lat_coords.max()
  lon_min, lon_max = lon_coords.min(), lon_coords.max()
  n_lat = len(lat_coords)
  n_lon = len(lon_coords)

  def latlon_to_pixel(lats, lons):
    px = (lons - lon_min) / (lon_max - lon_min) * (n_lon - 1)
    py = (lats - lat_min) / (lat_max - lat_min) * (n_lat - 1)
    return px, py

  def get_cyclone_positions(df, valid_time):
    frame_df = df[df["valid_time"] == valid_time]
    lons = frame_df["lon"].values % 360
    return frame_df["lat"].values, lons

  cols = min(cols, len(data))
  rows = math.ceil(len(data) / cols)
  figure = plt.figure(figsize=(plot_size * 2 * cols, plot_size * rows))
  figure.suptitle(fig_title, fontsize=16)
  figure.subplots_adjust(wspace=0, hspace=0)
  figure.tight_layout()

  images = []
  scatters = []
  for i, (title, (plot_data, norm, cmap)) in enumerate(data.items()):
    ax = figure.add_subplot(rows, cols, i+1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
    im = ax.imshow(
        plot_data.isel(time=0, missing_dims="ignore"), norm=norm,
        origin="lower", cmap=cmap)
    plt.colorbar(
        mappable=im, ax=ax, orientation="vertical", pad=0.02, aspect=16,
        shrink=0.75, cmap=cmap, extend=("both" if robust else "neither"))
    images.append(im)

    if cyclone_dfs and title in cyclone_dfs:
      lats_0, lons_0 = get_cyclone_positions(cyclone_dfs[title], valid_times[0])
      px, py = latlon_to_pixel(lats_0, lons_0)
      sc = ax.scatter(px, py, marker="x", c="red", s=20, linewidths=0.8, zorder=5)
      scatters.append((title, sc))
    else:
      scatters.append((title, None))

  def update(frame):
    if "time" in first_data.dims:
      td = datetime.timedelta(microseconds=first_data["time"][frame].item() / 1000)
      figure.suptitle(f"{fig_title}, {td}", fontsize=16)
    else:
      figure.suptitle(fig_title, fontsize=16)
    for im, (plot_data, norm, cmap) in zip(images, data.values()):
      im.set_data(plot_data.isel(time=frame, missing_dims="ignore"))
    for title, sc in scatters:
      if sc is not None:
        lats_f, lons_f = get_cyclone_positions(cyclone_dfs[title], valid_times[frame])
        offsets = (np.column_stack(latlon_to_pixel(lats_f, lons_f))
                   if len(lats_f) > 0 else np.empty((0, 2)))
        sc.set_offsets(offsets)

  ani = animation.FuncAnimation(fig=figure, func=update, frames=max_steps, interval=250)
  plt.close(figure.number)
  return HTML(ani.to_jshtml())


## Load the model (pretrained weights + config)

Same as the original demo, except we no longer load the bundled
`source-hres_forecast_init...nc` file as our meteorological input. We still
briefly load the *smallest* available bundled sample (`steps="01"`) purely to
harvest static, time-invariant fields (land-sea mask, orography, etc.) and
the exact dims/coords the model expects -- those don't depend on which date
you're forecasting, so it's fine to reuse them rather than trying to source
them from ERA5 separately.

In [ ]:
# @title Load model config + weights, and a small template for static fields/dims.

model_name = "WeatherNextCyclones_Mini"   # or WeatherNextCyclones_Mini + "_<2023" weights, see README
split = "2024"                             # "2024" -> trained through 2023; "2023" -> trained through 2022
data_resolution = "1.0"                    # must match the Mini model's native grid

config_name = f"weathernext2/configs/{model_name}"
weights_path = f"weathernext2/params/{model_name}_<{split}.npz"
# Smallest bundled sample, used ONLY for static fields + dims/coords template --
# NOT as the meteorological input (that comes from ERA5 below).
template_path = (f"weathernext2/dataset/source-hres_forecast_init-2024-10-07 "
                  f"00:00:00_res-{data_resolution}_levels-13_steps-01.nc")

gcs_client = storage.Client.create_anonymous_client()
gcs_bucket = gcs_client.get_bucket("dm_graphcast")

with gcs_bucket.blob(template_path).open("rb") as f:
  template_batch = xarray.load_dataset(f).compute()

config = fiddle_config_io.get_fiddle_config_by_name(config_name)
task_config = config.task

with gcs_bucket.blob(weights_path).open("rb") as f:
  ckpt = checkpoint.load(f, fgn.CheckPoint)

STATIC_VARS = [
    v for v in template_batch.data_vars
    if "time" not in template_batch[v].dims
]
print("Static/time-invariant fields reused from the template:", STATIC_VARS)


Static/time-invariant fields reused from the template: ['geopotential_at_surface', 'land_sea_mask']


In [ ]:
# @title Download IBTrACS (needed to pick a storm and seed the tracker).
try:
  ! wget -nc https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/netcdf/IBTrACS.ALL.v04r01.nc
  ds_ibtracs = xarray.open_dataset("IBTrACS.ALL.v04r01.nc", engine="h5netcdf").load()
  ds_ibtracs["time"] = ds_ibtracs["time"].dt.round("1s")
except Exception:
  print("Failed to download IBTrACS data. See https://www.ncei.noaa.gov/news/cloud-migration")
  ds_ibtracs = None


--2026-10-01 11:21:42--  https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/netcdf/IBTrACS.ALL.v04r01.nc
Resolving www.ncei.noaa.gov (www.ncei.noaa.gov)... 205.167.25.178, 205.167.25.177, 205.167.25.172, ...
Connecting to www.ncei.noaa.gov (www.ncei.noaa.gov)|205.167.25.178|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 23393268 (22M) [application/x-netcdf]
Saving to: ‘IBTrACS.ALL.v04r01.nc’

IBTrACS.ALL.v04r01. 100%[===================>]  22.31M  14.1MB/s    in 1.6s    

2026-10-01 11:21:44 (14.1 MB/s) - ‘IBTrACS.ALL.v04r01.nc’ saved [23393268/23393268]



## Choose the storm to reforecast

Set `STORM_SID` to an IBTrACS storm identifier (the `sid` field, e.g.
`"2022266N14283"`), or leave it as `None` and set
`INIT_TIME` directly to any 6-hourly UTC timestamp you want to initialize
from -- init doesn't have to be a storm's genesis time.

In [ ]:
# @title Pick a storm and its init time
STORM_SID = None                    # e.g. an IBTrACS sid string -- or leave None and set INIT_TIME manually
                                     # NOTE: ERA5 here only covers 1959-01-01 to 2023-01-10, so pick a
                                     # storm/date in that window (2022 Ian, 2019 Dorian, etc.).
INIT_TIME = pd.Timestamp("2021-02-27 00:00:00")  # Hurricane Ian genesis-ish; ERA5 (WB2) only covers 1959-2023

if STORM_SID is not None and ds_ibtracs is not None:
  storm = ds_ibtracs.sel(storm=(ds_ibtracs["sid"].astype(str) == STORM_SID))
  valid_times = pd.to_datetime(storm["time"].values.ravel())
  valid_times = valid_times[~pd.isna(valid_times)]
  INIT_TIME = pd.Timestamp(valid_times.min())  # genesis time

# ERA5 on WeatherBench2 only covers 1959 - early 2023; fail early with a
# clear message rather than a cryptic KeyError deep inside the zarr .sel().
if not (pd.Timestamp("1959-01-01") <= INIT_TIME <= pd.Timestamp("2022-01-10")):
  raise ValueError(
      f"init_time {INIT_TIME} is outside the WeatherBench2 ERA5 zarr's "
      f"coverage (1959-01-01 to 2023-01-10). Pick a different storm/date, "
      f"or point ERA5_ZARR at a dataset with the coverage you need.")

print(f"Reforecasting from init_time = {INIT_TIME} UTC")

all_storms_df, initial_storms_df = ibtracs_netcdf_to_csv.prepare_ibtracs_storms_dfs(
    ibtracs_ds=ds_ibtracs, init_time=INIT_TIME)


Reforecasting from init_time = 2021-02-27 00:00:00 UTC


/usr/local/lib/python3.13/dist-packages/weathernext/cyclones/ibtracs_netcdf_to_csv.py:172: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")


## Fetch ERA5 input + verification window

Pulls two input frames (`t-6h`, `t0=INIT_TIME`) plus `NUM_FORECAST_STEPS`
6-hourly frames *after* `INIT_TIME` from WeatherBench2's ERA5 zarr -- the
input frames feed the model, the later frames are real ERA5 truth used to
verify the forecast (this is a step up from the original demo, which
verified against the bundled sample's *own* forecast evolution rather than
independent truth).

ERA5 on this zarr covers 1959–2022; if you're reforecasting an older or very
recent storm, adjust `ERA5_ZARR` (or fall back to `hres_t0`, 2016+ only, for
recent dates -- see the format-compatibility discussion in chat).

In [ ]:
# @title Build the ERA5 input/verification batch

ERA5_ZARR = "gs://weatherbench2/datasets/era5/1959-2022-6h-1440x721.zarr"
NUM_FORECAST_STEPS = 60   # 20 * 6h = 5 days; must be <= what run_forward was compiled for
TARGET_RES_DEG = float(data_resolution)

def fetch_era5_window(init_time: pd.Timestamp, template: xarray.Dataset,
                       num_steps: int, static_vars: list[str]) -> xarray.Dataset:
  era5 = xarray.open_zarr(ERA5_ZARR, chunks=None, storage_options=dict(token="anon"))

  # time index 0 = t-6h, 1 = t0 (inputs); 2..num_steps+1 = target lead times.
  offsets_h = [-6, 0] + [6 * i for i in range(1, num_steps + 1)]
  valid_times = [init_time + pd.Timedelta(hours=h) for h in offsets_h]

  era5_time_index = pd.DatetimeIndex(era5.indexes["time"].to_datetimeindex()
                                      if hasattr(era5.indexes["time"], "to_datetimeindex")
                                      else era5.indexes["time"])
  era5_min, era5_max = era5_time_index.min(), era5_time_index.max()
  out_of_range = [t for t in valid_times if not (era5_min <= t <= era5_max)]
  if out_of_range:
    raise ValueError(
        f"{len(out_of_range)} requested timestamp(s) fall outside this ERA5 "
        f"zarr's coverage ({era5_min} to {era5_max}), e.g. {out_of_range[0]}. "
        f"Pick a storm/INIT_TIME within that range, or point ERA5_ZARR at a "
        f"dataset that covers it.")

  # Use nearest-match with a tight tolerance rather than exact equality --
  # exact list-based .sel() is brittle against calendar/precision quirks in
  # a given zarr store even when the date is logically in range. A mismatch
  # here (nearest match further than 1 step away) is a real data gap and
  # should still raise, which .sel(method="nearest") does via `tolerance`.
  missing = [t for t in valid_times if t not in era5_time_index]
  if missing:
    print(f"NOTE: {len(missing)} requested timestamp(s) not found exactly "
          f"in the ERA5 time index (e.g. {missing[0]}) -- snapping to "
          f"nearest available step within 1h.")

  time_varying_vars = [v for v in template.data_vars if v not in static_vars]
  # Two separate .sel calls: nearest-match tolerance is a time delta and
  # doesn't mix meaningfully with the level selector's units (hPa), so we
  # match time and level independently rather than in one combined .sel().
  ds = era5.sel(time=valid_times, method="nearest", tolerance=pd.Timedelta("1h"))
  ds = ds.sel(level=template.coords["level"].values)
  ds = ds[[v for v in time_varying_vars if v in ds.data_vars]]

  missing = set(time_varying_vars) - set(ds.data_vars)
  if missing:
    print(f"WARNING: variables not found in ERA5, check naming: {missing}")

  ds = ds.rename({"latitude": "lat", "longitude": "lon"})
  ds = ds.sortby("lat")  # ERA5 is lat-descending; also required before interp.

  ds = ds.isel(lat=slice(0, None, 4), lon=slice(0, None, 4))

  ds = ds.expand_dims(batch=[0])
  ds = ds.assign_coords(
      time=("time", np.array(offsets_h, dtype="timedelta64[h]").astype("timedelta64[ns]")),
  )
  # `datetime` must be 2D (batch, time), matching the original sample's
  # structure -- data_utils' forcing computation (day/year progress) expects
  # this shape and errors ("Number of feature dimensions (2) must be equal
  # to the number of data dimensions: 1") if datetime is only 1D over time.
  datetime_2d = np.broadcast_to(
      np.array(valid_times, dtype="datetime64[ns]"), (1, len(valid_times)))
  ds = ds.assign_coords(datetime=(("batch", "time"), datetime_2d))
  ds = ds.transpose(*[d for d in template.dims if d in ds.dims], ...)

  for v in static_vars:
    ds[v] = template[v]

  still_missing = [v for v in time_varying_vars if v not in ds.data_vars]
  for v in still_missing:
    tmpl_var = template[v]
    dim_sizes = []
    for d in tmpl_var.dims:
      if d == "time":
        dim_sizes.append(len(valid_times))
      elif d in ds.sizes:
        dim_sizes.append(ds.sizes[d])
      else:
        dim_sizes.append(tmpl_var.sizes[d])
    ds[v] = (tmpl_var.dims, np.full(dim_sizes, np.nan, dtype=np.float32))
  if still_missing:
    print(f"NOTE: filled these with NaN placeholders (not in ERA5): {still_missing}")

  return ds.compute()


era5_batch = fetch_era5_window(INIT_TIME, template_batch, NUM_FORECAST_STEPS, STATIC_VARS)
print(era5_batch.dims.mapping)


NOTE: filled these with NaN placeholders (not in ERA5): ['cyclone_usa_pres_disc', 'cyclone_usa_r64_nw_radius_disc', 'cyclone_usa_r50_ne_radius_disc', 'cyclone_all_wind_disc', 'cyclone_usa_r64_sw_radius_disc', 'seconds_since_epoch', 'cyclone_usa_rmw_disc', 'year_progress_sin', 'cyclone_usa_r64_ne_radius_disc', 'cyclone_usa_r50_sw_radius_disc', 'day_progress_cos', 'cyclone_usa_r50_nw_radius_disc', 'year_progress_cos', 'cyclone_usa_r34_ne_radius_disc', 'cyclone_usa_r34_se_radius_disc', 'cyclone_usa_r34_sw_radius_disc', 'cyclone_usa_r50_se_radius_disc', 'cyclone_exists_gaussian_unit_mode', 'cyclone_usa_r64_se_radius_disc', 'day_progress_sin', 'cyclone_usa_r34_nw_radius_disc', 'cyclone_usa_wind_disc']
{'batch': 1, 'time': 62, 'lat': 181, 'lon': 360, 'level': 13}


In [ ]:
era5_batch


<xarray.Dataset> Size: 2GB
Dimensions:                            (batch: 1, time: 62, lat: 181, lon: 360,
                                        level: 13)
Coordinates:
  * batch                              (batch) int64 8B 0
  * time                               (time) timedelta64[ns] 496B -1 days +1...
    datetime                           (batch, time) datetime64[ns] 496B 2021...
  * lat                                (lat) float32 724B -90.0 -89.0 ... 90.0
  * lon                                (lon) float32 1kB 0.0 1.0 ... 358.0 359.0
  * level                              (level) int64 104B 50 100 ... 925 1000
Data variables: (12/36)
    specific_humidity                  (batch, time, lat, lon, level) float32 210MB ...
    10m_v_component_of_wind            (batch, time, lat, lon) float32 16MB -...
    v_component_of_wind                (batch, time, lat, lon, level) float32 210MB ...
    2m_temperature                     (batch, time, lat, lon) float32 16MB 2...
    10m_u_component_of_wind            (batch, time, lat, lon) float32 16MB 0...
    vertical_velocity                  (batch, time, lat, lon, level) float32 210MB ...
    ...                                 ...
    cyclone_usa_r50_se_radius_disc     (batch, time, lat, lon) float32 16MB n...
    cyclone_exists_gaussian_unit_mode  (batch, time, lat, lon) float32 16MB n...
    cyclone_usa_r64_se_radius_disc     (batch, time, lat, lon) float32 16MB n...
    day_progress_sin                   (batch, time, lon) float32 89kB nan .....
    cyclone_usa_r34_nw_radius_disc     (batch, time, lat, lon) float32 16MB n...
    cyclone_usa_wind_disc              (batch, time, lat, lon) float32 16MB n...

## Extract inputs/targets/forcings

No `train_*` split here -- training cells were removed since we're only
running inference with the pretrained checkpoint.

In [ ]:
# @title Extract eval inputs/targets/forcings from the ERA5 batch

eval_inputs, eval_targets, eval_forcings = data_utils.extract_inputs_targets_forcings(
    era5_batch, target_lead_times=slice("6h", f"{NUM_FORECAST_STEPS * 6}h"),
    **dataclasses.asdict(task_config))

print("Eval Inputs:   ", eval_inputs.dims.mapping)
print("Eval Targets:  ", eval_targets.dims.mapping)
print("Eval Forcings: ", eval_forcings.dims.mapping)


Eval Inputs:    {'batch': 1, 'time': 2, 'lat': 181, 'lon': 360, 'level': 13}
Eval Targets:   {'batch': 1, 'time': 60, 'lat': 181, 'lon': 360, 'level': 13}
Eval Forcings:  {'batch': 1, 'time': 60, 'lon': 360}


In [ ]:
# @title Build jitted inference function (loss/grad training cells removed)

backend = jax.default_backend()
transformer_kwargs = config.predictor_kwargs['noisy_function_kwargs'][
    'mesh_model_ctor'
].keywords['transformer_kwargs']
if backend == 'gpu':
  transformer_kwargs['attention_type'] = 'triblockdiag_mha'
elif backend == 'tpu':
  transformer_kwargs.update({
      'block_q': 128, 'block_kv': 128, 'block_kv_compute': 128,
      'block_q_dkv': 128, 'block_kv_dkv': 128, 'block_kv_dkv_compute': 128,
  })

config_inference = fgn.PredictorConfig(
    task=config.task,
    predictor_constructor=config.predictor_constructor,
    predictor_kwargs=config.predictor_kwargs,
    predictor_wrappers=config.predictor_wrappers[:-1],  # Remove ensemble wrapper
)

@hk.transform
def run_forward(inputs, targets_template, forcings):
  predictor = fgn.construct_predictor(config_inference)
  return predictor(inputs, targets_template=targets_template, forcings=forcings)

run_forward_jitted = jax.jit(
    lambda rng, i, t, f: run_forward.apply(ckpt.params, rng, i, t, f)
)
run_forward_pmap = xarray_jax.pmap(run_forward_jitted, dim="sample")


## Run the model

In [ ]:
# The number of ensemble members should be a multiple of the number of devices.
print(f"Number of local devices {len(jax.local_devices())}")


Number of local devices 1


In [ ]:
import gc

for _name in ("predictions", "chunks", "data_to_track", "predicted_ensemble_tracks"):
  if _name in globals():
    del globals()[_name]

gc.collect()
jax.clear_caches()


In [ ]:
# # @title Autoregressive rollout (loop in python)

# num_ensemble_members = 5 # @param int
# rng = jax.random.PRNGKey(0)
# rngs = np.stack(
#     [jax.random.fold_in(rng, i) for i in range(num_ensemble_members)], axis=0)

# chunks = []
# for chunk in rollout.chunked_prediction_generator_multiple_runs(
#     predictor_fn=run_forward_pmap,
#     rngs=rngs,
#     inputs=eval_inputs,
#     targets_template=eval_targets * np.nan,
#     forcings=eval_forcings,
#     num_steps_per_chunk=1,
#     num_samples=num_ensemble_members,
#     pmap_devices=jax.local_devices()
#     ):
#     chunks.append(jax.device_get(chunk))
# predictions = xarray.combine_by_coords(chunks)


## Run the direct tracker

Same purpose-built cyclone extraction the original demo uses -- no
TempestExtremes needed, this is the model's own tracking code.

In [ ]:
# # @title Run the tracker
tracker_config = direct_tracker_6h_v1_config.get_config()
tracker = tracker_config.tracker_constructor(**tracker_config.tracker_kwargs)

# predicted_ensemble_tracks = []
# for sample_idx in predictions.sample:
#   data_to_track = predictions.isel(batch=0).sel(sample=sample_idx)
#   data_to_track = tracker.preprocess_gridded_ds(data_to_track)
#   data_to_track = data_to_track.expand_dims(forecast_datetime=[INIT_TIME])
#   data_to_track = data_to_track.assign_coords(
#       lead_time_secs=data_to_track.time.astype("timedelta64[s]").astype(int)
#   )
#   data_to_track = data_to_track.assign_coords(
#       date_time=data_to_track.forecast_datetime.data + data_to_track.time
#   )
#   data_to_track = data_to_track.as_numpy()

#   predicted_tracks = tracker(
#       gridded_ds=data_to_track,
#       initial_storms_df=initial_storms_df,
#       do_cyclogenesis=True,
#   )
#   predicted_tracks[cyclone_constants.LON] = (
#       predicted_tracks[cyclone_constants.LON] % 360
#   )
#   predicted_ensemble_tracks.append(predicted_tracks)


In [ ]:
import pandas as pd

# # Combine the list of DataFrames and assign the list index (0, 1, 2, ...) as a key
# df_all = pd.concat(
#     predicted_ensemble_tracks, keys=range(len(predicted_ensemble_tracks))
# )

# # Move the key index into its own column named 'ensemble_member' (or 'array_id')
# df_all = df_all.reset_index(level=0).rename(
#     columns={"level_0": "ensemble_member"}
# )

# # Save to CSV
# df_all.to_csv("predicted_ensemble_tracks.csv", index=False)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import traceback

MANIFEST_PATH = "/content/drive/MyDrive/wn2_au_reforecast/manifest.csv"
TRACKS_DIR = "/content/drive/MyDrive/wn2_au_reforecast/tracks"
os.makedirs(TRACKS_DIR, exist_ok=True)
os.makedirs(os.path.dirname(MANIFEST_PATH), exist_ok=True)

BATCH_SIZE = 15                # proven-stable ensemble size per rollout call
TARGET_ENSEMBLE_TOTAL = 30      # total members wanted per storm
MIN_SEASON, MAX_SEASON = 1980, 2021   # post-2022 handled by your other script


Mounted at /content/drive


In [ ]:
def decode_scalar(val):
  item = val.item() if hasattr(val, "item") else val
  return item.decode() if isinstance(item, bytes) else str(item)


def build_australia_manifest(ds_ibtracs, min_season, max_season):
  records = []
  for i in range(ds_ibtracs.sizes["storm"]):
    storm = ds_ibtracs.isel(storm=i)
    valid_mask = ~pd.isna(storm["time"].values)
    if not valid_mask.any():
      continue
    first_idx = int(np.where(valid_mask)[0][0])
    genesis_time = pd.Timestamp(storm["time"].values[first_idx])

    basin_str = decode_scalar(storm["basin"].values[first_idx]).strip()
    if basin_str not in ("SI", "SP"):
      continue

    season = int(storm["season"].values) if "season" in storm else genesis_time.year
    if not (min_season <= season <= max_season):
      continue

    records.append({
        "sid": decode_scalar(storm["sid"].values),
        "season": season,
        "genesis_time": genesis_time.isoformat(),
        "status": "pending",
        "ensembles_done": 0,
        "error": "",
    })

  manifest = pd.DataFrame.from_records(records)
  return manifest.sort_values("genesis_time", ascending=False).reset_index(drop=True)


def load_or_create_manifest():
  if os.path.exists(MANIFEST_PATH):
    return pd.read_csv(MANIFEST_PATH)
  manifest = build_australia_manifest(ds_ibtracs, MIN_SEASON, MAX_SEASON)
  manifest.to_csv(MANIFEST_PATH, index=False)
  return manifest


def save_manifest(manifest):
  tmp = MANIFEST_PATH + ".tmp"
  manifest.to_csv(tmp, index=False)
  os.replace(tmp, MANIFEST_PATH)


def append_tracks(storm_id, new_tracks_df):
  path = os.path.join(TRACKS_DIR, f"{storm_id}.csv")
  if os.path.exists(path):
    new_tracks_df = pd.concat([pd.read_csv(path), new_tracks_df], ignore_index=True)
  new_tracks_df.to_csv(path, index=False)


In [ ]:
def run_one_batch(genesis_time, ensemble_offset, batch_size):
  init_time = pd.Timestamp(genesis_time)
  era5_batch = fetch_era5_window(init_time, template_batch, NUM_FORECAST_STEPS, STATIC_VARS)
  eval_inputs, eval_targets, eval_forcings = data_utils.extract_inputs_targets_forcings(
      era5_batch, target_lead_times=slice("6h", f"{NUM_FORECAST_STEPS * 6}h"),
      **dataclasses.asdict(task_config))
  _, initial_storms_df = ibtracs_netcdf_to_csv.prepare_ibtracs_storms_dfs(
      ibtracs_ds=ds_ibtracs, init_time=init_time)

  rng = jax.random.PRNGKey(0)
  rngs = np.stack([jax.random.fold_in(rng, ensemble_offset + i) for i in range(batch_size)], axis=0)

  chunks = []
  for chunk in rollout.chunked_prediction_generator_multiple_runs(
      predictor_fn=run_forward_pmap, rngs=rngs, inputs=eval_inputs,
      targets_template=eval_targets * np.nan, forcings=eval_forcings,
      num_steps_per_chunk=1, num_samples=batch_size, pmap_devices=jax.local_devices()):
    chunks.append(jax.device_get(chunk))
  predictions = xarray.combine_by_coords(chunks)

  batch_tracks = []
  for local_idx, sample_idx in enumerate(predictions.sample.values):
    data_to_track = predictions.isel(batch=0).sel(sample=sample_idx)
    data_to_track = tracker.preprocess_gridded_ds(data_to_track)
    data_to_track = data_to_track.expand_dims(forecast_datetime=[init_time])
    data_to_track = data_to_track.assign_coords(
        lead_time_secs=data_to_track.time.astype("timedelta64[s]").astype(int))
    data_to_track = data_to_track.assign_coords(
        date_time=data_to_track.forecast_datetime.data + data_to_track.time)
    data_to_track = data_to_track.as_numpy()

    tracks = tracker(gridded_ds=data_to_track, initial_storms_df=initial_storms_df, do_cyclogenesis=True)
    tracks[cyclone_constants.LON] = tracks[cyclone_constants.LON] % 360
    tracks["ensemble_member"] = ensemble_offset + local_idx
    batch_tracks.append(tracks)

  batch_df = pd.concat(batch_tracks, ignore_index=True)

  # del predictions, chunks
  # gc.collect()
  # jax.clear_caches()

  return batch_df


In [ ]:
manifest = load_or_create_manifest()
pending = manifest[manifest["status"] != "done"]
print(f"{len(pending)} / {len(manifest)} storms remaining (newest first)")

for idx in pending.index:
  row = manifest.loc[idx]
  storm_id, season, genesis_time = row["sid"], row["season"], row["genesis_time"]
  print(f"=== {storm_id} (season {season}) ===")
  try:
    ensembles_done = int(row["ensembles_done"])
    while ensembles_done < TARGET_ENSEMBLE_TOTAL:
      this_batch = min(BATCH_SIZE, TARGET_ENSEMBLE_TOTAL - ensembles_done)
      batch_df = run_one_batch(genesis_time, ensemble_offset=ensembles_done, batch_size=this_batch)
      append_tracks(storm_id, batch_df)
      ensembles_done += this_batch
      manifest.loc[idx, "ensembles_done"] = ensembles_done
      save_manifest(manifest)
      print(f"  {ensembles_done}/{TARGET_ENSEMBLE_TOTAL} members done")
    manifest.loc[idx, "status"] = "done"

  except Exception as e:
    manifest.loc[idx, "status"] = "failed"
    manifest.loc[idx, "error"] = f"{type(e).__name__}: {e}"
    print(f"  FAILED: {e}")
    traceback.print_exc()

  finally:
    save_manifest(manifest)

print("Run complete (or interrupted) -- re-run this cell any time to resume.")
